In [ ]:
#| default_exp probe

In [ ]:
#| hide
import json
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from fastcore.all import Path

Probe native-window support, compatible interpreters, HTTP readiness, and running application processes.

In [ ]:
#| export
from __future__ import annotations
import json, os, subprocess, sys, sysconfig, time
from fastcore.all import Path, first

In [ ]:
#| export
BACKENDS = {'darwin': 'cocoa', 'win32': 'edgechromium', 'linux': 'gtk'}

FRAMEWORK_CANDIDATES = (
    '/Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13',
    '/Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14',
    '/Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12',
    '/opt/homebrew/opt/python@3.13/bin/python3.13',
    '/opt/homebrew/opt/python@3.14/bin/python3.14',
    '/opt/homebrew/opt/python@3.12/bin/python3.12',
    '/usr/local/opt/python@3.13/bin/python3.13',
    '/usr/local/opt/python@3.14/bin/python3.14',
    '/usr/local/opt/python@3.12/bin/python3.12',
    '/usr/bin/python3',
)

`BACKENDS` maps supported platforms to pywebview GUI names. macOS bundles require a framework build; standalone uv and pyenv interpreters are excluded.

In [ ]:
#| export
def backend(platform=None):
    "pywebview's GUI name for `platform`, or None where there is no supported webview."
    return BACKENDS.get(platform or sys.platform)

def shell_ready(platform=None):
    "`(ok, why)`: whether a native window can be opened here, and what is missing if not."
    gui = backend(platform)
    if gui is None: return False, f'no native webview backend for {platform or sys.platform}'
    try: import webview  # noqa: F401
    except ImportError as e: return False, f'pywebview is not installed ({e})'
    return True, gui

`shell_ready` returns native backend availability and the selected pywebview GUI name.

In [ ]:
backend('darwin'), backend('linux'), backend('plan9')

('cocoa', 'gtk', None)

In [ ]:
test_eq(shell_ready('plan9'), (False, 'no native webview backend for plan9'))
ok, why = shell_ready('darwin')
test_eq(ok, why == 'cocoa')

In [ ]:
#| export
def py_version(python):
    "The `(major, minor)` of `python`, or None when it will not say."
    try: out = subprocess.run([str(python), '-c', 'import sys;print(*sys.version_info[:2])'],
                              capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return None
    try: return tuple(int(n) for n in out.stdout.split()) or None
    except ValueError: return None

def is_framework(python=None):
    "Whether `python` (default: this interpreter) is a macOS framework build py2app can use."
    if python is None: return bool(sysconfig.get_config_var('PYTHONFRAMEWORK'))
    probe = ('import json,sysconfig,sys;'
             "print(json.dumps([sysconfig.get_config_var('PYTHONFRAMEWORK') or '', "
             'list(sys.version_info[:2])]))')
    try: out = subprocess.run([str(python), '-c', probe], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError): return False
    if out.returncode: return False
    try: name, version = json.loads(out.stdout.strip() or 'null')
    except (ValueError, TypeError): return False
    return bool(name) and tuple(version) >= (3, 12)

def framework_python(candidates=FRAMEWORK_CANDIDATES):
    "The first framework interpreter on this machine that py2app can build against."
    return first(p for c in candidates if (p := Path(c)).exists() and is_framework(p))

`py_version` and `is_framework` inspect another interpreter without raising when it cannot run.

In [ ]:
#| hide
tmp = TemporaryDirectory(); tdir = Path(tmp.name)
def fake_py(name, says):
    "An executable answering a probe the way an interpreter would, saying whatever `says` says."
    p = tdir/name; p.write_text(f'#!/bin/sh\necho {says!r}\n'); p.chmod(0o755)
    return p
fw  = fake_py('framework3.13',  json.dumps(['Python', [3, 13]]))
old = fake_py('framework3.11',  json.dumps(['Python', [3, 11]]))
pbs = fake_py('standalone3.13', json.dumps(['', [3, 13]]))
mute, chatty = fake_py('mute', ''), fake_py('chatty', 'no version here')

Three interpreters that answer the probe differently: a framework build py2app can embed, one on a
Python too old, and a python-build-standalone whose `PYTHONFRAMEWORK` is empty. The last is what uv
and pyenv install.

In [ ]:
is_framework(fw), is_framework(old), is_framework(pbs)

(True, False, False)

In [ ]:
#| hide
test_eq(framework_python([tdir/'nothing', pbs, old, fw]), fw)
test_eq(framework_python([pbs, old]), None)
test_eq(framework_python([]), None)
test_eq(py_version(sys.executable), tuple(sys.version_info[:2]))
test_eq(py_version(tdir/'nothing'), None)
test_eq(py_version(chatty), None)
test_eq(py_version(mute), None)  

In [ ]:
#| export
def wait_for_http(url, timeout=60, interval=.1):
    "Block until `url` answers, or `timeout` passes. True if the server came up."
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except Exception: time.sleep(interval)
    return False

`wait_for_http` polls until the server responds or the timeout expires. Connection errors return `False` rather than raising.

In [ ]:
#| hide
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Thread
class _Root(BaseHTTPRequestHandler):
    def do_GET(self): self.send_response(200 if self.path == '/' else 404); self.end_headers()
    def log_message(self, *a): pass
srv = HTTPServer(('127.0.0.1', 0), _Root); Thread(target=srv.serve_forever, daemon=True).start()
url = f'http://127.0.0.1:{srv.server_port}/'

In [ ]:
test_eq(wait_for_http(url), True)
test_eq(wait_for_http(url + 'missing', timeout=.3, interval=.05), False)

In [ ]:
#| export
def running_from(bundle, ps_output=None):
    "PIDs of processes whose executable is inside `bundle`."
    target = str(Path(bundle).resolve())
    if ps_output is None:
        try: ps_output = subprocess.run(['ps', '-Ao', 'pid,command'], capture_output=True,
                                        text=True, timeout=10).stdout
        except (OSError, subprocess.SubprocessError): return []
    pids = []
    for line in ps_output.splitlines()[1:]:
        pid, _, command = line.strip().partition(' ')
        exe = command.split(' ', 1)[0]
        if exe.startswith(target + os.sep) and pid.isdigit() and int(pid) != os.getpid():
            pids.append(int(pid))
    return pids

`running_from` finds matching processes from supplied or live `ps` output. Process-list failures return an empty list.

In [ ]:
ps = """  PID COMMAND
  101 /Apps/Demo.app/Contents/MacOS/Demo --serve
  102 /usr/bin/grep -r /Apps/Demo.app
  103 /Apps/Demo.app.old/Contents/MacOS/Demo
  104 /bin/zsh
"""
running_from('/Apps/Demo.app', ps)

[101]

In [ ]:
#| hide
test_eq(running_from('/Apps/Demo.app', f'PID COMMAND\n  {os.getpid()} /Apps/Demo.app/MacOS/Demo\n'), [])
test_eq(running_from('/Apps/Demo.app', ''), [])
test_eq(running_from('/Apps/Demo.app', '  101 /Apps/Demo.app/MacOS/Demo\n'), [])

In [ ]:
#| hide
srv.shutdown(); tmp.cleanup()